### BINDING NUMERI CON NOMI

In [ ]:
import re

def extract_and_replace_numbers(chat_text):
    # Pattern per numeri internazionali WhatsApp
    formatted_pattern = r"\+\d{2,3}(?: \d+){1,3}"
    
    # Trova numeri non formattati
    plain_pattern = r"(?<!\d)([3-5]\d{10,12})(?!\d)"
    
    # Trova tutti i numeri nei due formati
    formatted_numbers = re.findall(formatted_pattern, chat_text)
    plain_numbers = re.findall(plain_pattern, chat_text)
    
    numbers = formatted_numbers + plain_numbers

    numbers_nodup = set(num for num in numbers)
    
    return numbers_nodup
#rimuove spazi e ritorna solo cifre
def normalize_number(n):
    return re.sub(r"[^\d]", "", n)

# Ritorna un dizionario: numero originale → 'Company Employee 1', 'Support Technician 1', ... due numeri che normalizzati sono uguali hanno la stessa etichetta.
def classify_numbers_with_roles(numbers):
    classification_norm = {}
    company_count = 1
    tech_count = 1

    # Primo assegna etichette ai numeri normalizzati (unici)
    normalized_numbers = sorted(set(normalize_number(n) for n in numbers))
    for norm in normalized_numbers:

        # divisione tra numeri italiani e non italiani
        if norm.startswith("39"):  # Italia
            label = f"Company Employee {company_count}"
            company_count += 1
        else:
            label = f"Support Technician {tech_count}"
            tech_count += 1
        classification_norm[norm] = label

    # Ora mappa ogni numero originale alla sua etichetta basata sul numero normalizzato
    classification = {}
    for num in numbers:
        norm = normalize_number(num)
        classification[num] = classification_norm[norm]

    return classification

def replace_numbers(chat_text, mapping):
    # Ordina le chiavi per lunghezza decrescente per evitare sostituzioni parziali
    for original in sorted(mapping, key=len, reverse=True):
        label = mapping[original]
        # Escape per evitare errori nei caratteri speciali (come '+')
        chat_text = re.sub(re.escape(original), label, chat_text)
    return chat_text

def replace_numbers_atsimbol(chat_text, mapping):
    # Ordina le chiavi per lunghezza decrescente per evitare sostituzioni parziali
    for original in sorted(mapping, key=len, reverse=True):
        label = "@" + mapping[original]
        # Escape per evitare errori nei caratteri speciali (come '+')
        chat_text = re.sub(re.escape(original), label, chat_text)
    return chat_text


with open("../ChatSSI.txt", encoding="utf-8") as f:
    chat = f.read()

# Estrai numeri e classificali
numbers = extract_and_replace_numbers(chat)
print("numeri trovati")
people = classify_numbers_with_roles(numbers)
print("persone mappate")

# Sostituisci i numeri nel testo
replaced_chat = replace_numbers(chat, people)
print("sostituiti nel testo")
replaced_chat = replace_numbers_atsimbol(replaced_chat, people)
print("sostituiti nel @")
# Salva il file modificato
with open("../ChatSSI_clean.txt", "w", encoding="utf-8") as f_out:
    f_out.write(replaced_chat)

print("Done")

### CLAP DEI SINGOLI MESSAGGI + DF

In [ ]:
import re
import pandas as pd
from datetime import datetime, timedelta

clean_chat = "ChatSSI_clean.txt"


with open("../" + clean_chat, encoding="utf-8") as f:
    lines = f.readlines()

# uniamo messaggi multilinea
messages = []
current_message = ""

# pattern per detectare l'inizio di un nuovo messaggio
timestamp_pattern = re.compile(r"^\d{2}/\d{2}/\d{2}, \d{2}:\d{2} - ")

for line in lines:
    if timestamp_pattern.match(line):
        if current_message:
            messages.append(current_message.strip())
        current_message = line.strip()
    else:
        current_message += " " + line.strip()

if current_message:
    messages.append(current_message.strip())

# estraiamo timestamp, sender e message
parsed_data = []
message_pattern = re.compile(r"^(\d{2}/\d{2}/\d{2}), (\d{2}:\d{2}) - (.*?): (.*)$")

for msg in messages:
    match = message_pattern.match(msg)
    if match:
        date, time, sender, text = match.groups()
        dt = datetime.strptime(f"{date} {time}", "%d/%m/%y %H:%M")
        parsed_data.append((dt, sender.strip(), text.strip()))
    else:
        continue


df = pd.DataFrame(parsed_data, columns=["datetime", "sender", "message"])
df = df.sort_values("datetime") #non cancellare per sicurezza
df.to_csv("../ChatSSI_clean.csv", index=False)

df.head()

### Test per vedere perché nel csv ci sono meno righe rispetto al txt

In [ ]:

import re

clean_chat = "ChatSSI_clean.txt"

with open("../" + clean_chat, encoding="utf-8") as f:
    lines = f.readlines()

# Pattern per identificare l'inizio di un messaggio
timestamp_pattern = re.compile(r"^\d{2}/\d{2}/\d{2}, \d{2}:\d{2} - ")

tot_righe = len(lines)
messaggi_con_timestamp = 0
righe_multilinea = 0

for line in lines:
    if timestamp_pattern.match(line):
        messaggi_con_timestamp += 1
    else:
        # Riga che non inizia con data/ora, quindi è l'utente che è andato a capo
        righe_multilinea += 1

print(f"Totale righe nel file .txt: {tot_righe}")
print(f"Righe che iniziano con timestamp (nuovi messaggi): {messaggi_con_timestamp}")
print(f"Righe accorpate (a capo nei messaggi multilinea): {righe_multilinea}")

In [ ]:
import tiktoken

txt = "ChatSSI_clean.txt"


with open("../" + txt, encoding="utf-8") as f:
    text = f.read()


encoding = tiktoken.get_encoding("cl100k_base")

#29/12/21
#22/12/22
#27/12/23
#20/12/24
#resto

#text = text.split("19/12/24")[-1]
#text = text.split("20/12/24")[0]


tokens = encoding.encode(text)
numero_di_token = len(tokens)

print(f"{numero_di_token} token")

In [ ]:
import os
from PIL import Image

image_dir = '../chat_images'
total_pixels = 0
image_count = 0

for filename in os.listdir(image_dir):
    if filename.lower().endswith(('.jpg', '.jpeg')):
        filepath = os.path.join(image_dir, filename)
        try:
            with Image.open(filepath) as img:
                width, height = img.size
                pixels = width * height
                total_pixels += pixels
                image_count += 1
        except Exception as e:
            print(f"Errore nell'apertura di {filename}: {e}")

if image_count > 0:
    average_pixels = total_pixels / image_count
    print(f"Immagini elaborate: {image_count}")
    print(f"Media di pixel per immagine: {average_pixels:.2f}")
else:
    print("Nessuna immagine JPG trovata nella cartella.")

In [ ]:
with open('../Chats/2025_clean.txt', 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if '(file allegato' in line and '.jpg (file allegato' not in line:
            print(f'Line {i+1}: {line.strip()}')